In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mikoajkolman/pokemon-images-first-generation17000-files")

print(path)

C:\Users\Miguel\.cache\kagglehub\datasets\mikoajkolman\pokemon-images-first-generation17000-files\versions\1


Identificando as classes de acordo com as pastas do dataset

In [2]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from pathlib import Path

path = Path(path)
pokemon_path = path / "pokemon"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  
])

dataset = ImageFolder(
    pokemon_path, transform=transform)


print(dataset.classes)

['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine', 'Articuno', 'Beedrill', 'Bellsprout', 'Blastoise', 'Bulbasaur', 'Butterfree', 'Caterpie', 'Chansey', 'Charizard', 'Charmander', 'Charmeleon', 'Clefable', 'Clefairy', 'Cloyster', 'Cubone', 'Dewgong', 'Diglett', 'Ditto', 'Dodrio', 'Doduo', 'Dragonair', 'Dragonite', 'Dratini', 'Drowzee', 'Dugtrio', 'Eevee', 'Ekans', 'Electabuzz', 'Electrode', 'Exeggcute', 'Exeggutor', 'Farfetchd', 'Fearow', 'Flareon', 'Gastly', 'Gengar', 'Geodude', 'Gloom', 'Golbat', 'Goldeen', 'Golduck', 'Graveler', 'Grimer', 'Growlithe', 'Gyarados', 'Haunter', 'Hitmonchan', 'Hitmonlee', 'Horsea', 'Hypno', 'Ivysaur', 'Jigglypuff', 'Jolteon', 'Jynx', 'Kabutops', 'Kadabra', 'Kakuna', 'Kangaskhan', 'Kingler', 'Koffing', 'Lapras', 'Lickitung', 'Machamp', 'Machoke', 'Machop', 'Magikarp', 'Magmar', 'Magnemite', 'Magneton', 'Mankey', 'Marowak', 'Meowth', 'Metapod', 'Mew', 'Mewtwo', 'Moltres', 'Mr. Mime', 'MrMime', 'Nidoking', 'Nidoqueen', 'Nidorina', 'Nidorino', 'Ninetales

In [3]:
from torch.utils.data import Dataset

class FilteredDataset(Dataset):

    def __init__(self, dataset, classes):
        self.dataset = dataset
        self.classes = classes

        self.class_to_idx = {name:i for i, name in enumerate(classes)}

        original_idx = {dataset.class_to_idx[name]: self.class_to_idx[name]for name in classes}

        self.samples = []

        for path, label in dataset.samples:
                if label in original_idx:
                    novo_label = original_idx[label]
                    self.samples.append((path, novo_label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        image = self.dataset.loader(path)

        if self.dataset.transform:
            image = self.dataset.transform(image)

        return image, label

In [16]:
classes = ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine', 'Articuno', 'Beedrill', 'Bellsprout', 'Blastoise', 'Bulbasaur', 'Butterfree', 'Caterpie', 'Chansey', 'Charizard', 'Charmander', 'Charmeleon', 'Clefable', 'Clefairy', 'Cloyster', 'Cubone', 'Dewgong', 'Diglett', 'Ditto', 'Dodrio', 'Doduo', 'Dragonair', 'Dragonite', 'Dratini', 'Drowzee', 'Dugtrio', 'Eevee', 'Ekans', 'Electabuzz', 'Electrode', 'Exeggcute', 'Exeggutor', 'Farfetchd', 'Fearow', 'Flareon', 'Gastly', 'Gengar', 'Geodude', 'Gloom', 'Golbat', 'Goldeen', 'Golduck', 'Graveler', 'Grimer', 'Growlithe', 'Gyarados', 'Haunter', 'Hitmonchan', 'Hitmonlee', 'Horsea', 'Hypno', 'Ivysaur', 'Jigglypuff', 'Jolteon', 'Jynx', 'Kabutops', 'Kadabra', 'Kakuna', 'Kangaskhan', 'Kingler', 'Koffing', 'Lapras', 'Lickitung', 'Machamp', 'Machoke', 'Machop', 'Magikarp', 'Magmar']
dataset = FilteredDataset(dataset, classes)

In [5]:
from torch.utils.data import random_split
total = len(dataset)

train_size =int(0.7*total)
val_size = int(0.15*total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

In [6]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Visualizar batch

In [7]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT
)

In [8]:
num_classes = len(dataset.classes)

print(num_classes)

72


In [9]:
import torch.nn as nn

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

In [10]:
import torch 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
def accuracy(outputs, labels):
    _, predicted = outputs.max(1)
    correct = (predicted == labels).sum().item()
    return correct / labels.size(0)

In [12]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [13]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, name_model="model"):
    best_acc =0.0
    
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print('-' * 20)

        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct_train / total_train
        print(f"Training Loss: {epoch_loss:.4f} Accuracy: {epoch_acc:.4f}")
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)
        
        
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_val / total_val
        print(f"Validation Loss: {epoch_val_loss:.4f} Accuracy: {epoch_val_acc:.4f}")

        if epoch_val_acc > best_acc:
            torch.save(model.state_dict(), f"../models/classifier/{name_model}_epoch_{epoch}.pth")
            best_acc = epoch_val_acc
            
        

In [14]:
train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)

Epoch 1/10
--------------------


C:\Users\Miguel\AppData\Roaming\Python\Python312\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Training Loss: 1.2716 Accuracy: 0.6800
Validation Loss: 0.8400 Accuracy: 0.7776
Epoch 2/10
--------------------
Training Loss: 0.4869 Accuracy: 0.8636
Validation Loss: 0.9068 Accuracy: 0.7542
Epoch 3/10
--------------------
Training Loss: 0.3233 Accuracy: 0.9110
Validation Loss: 0.5899 Accuracy: 0.8395
Epoch 4/10
--------------------
Training Loss: 0.2388 Accuracy: 0.9305
Validation Loss: 0.3528 Accuracy: 0.9047
Epoch 5/10
--------------------
Training Loss: 0.1992 Accuracy: 0.9440
Validation Loss: 0.6401 Accuracy: 0.8244
Epoch 6/10
--------------------
Training Loss: 0.2160 Accuracy: 0.9345
Validation Loss: 0.5542 Accuracy: 0.8512
Epoch 7/10
--------------------
Training Loss: 0.1912 Accuracy: 0.9438
Validation Loss: 0.5365 Accuracy: 0.8545
Epoch 8/10
--------------------
Training Loss: 0.1563 Accuracy: 0.9531
Validation Loss: 0.4558 Accuracy: 0.8813
Epoch 9/10
--------------------
Training Loss: 0.1199 Accuracy: 0.9647
Validation Loss: 0.4149 Accuracy: 0.8955
Epoch 10/10
------------

In [19]:
from PIL import Image

img = Image.open("../images/Charmander.png").convert("RGB")

img = transform(img)
img = img.unsqueeze(0)

with torch.no_grad():
    output = model(img)

probabilities = torch.softmax(output, dim=1)

prediction = probabilities.argmax(1).item()

confidence = probabilities[0][prediction].item()

for classe, prob in zip(classes, probabilities[0]):
    print(f"{classe:15} {prob.item():.2%}")

Abra            0.00%
Aerodactyl      0.00%
Alakazam        0.00%
Arbok           0.00%
Arcanine        0.00%
Articuno        0.00%
Beedrill        0.00%
Bellsprout      0.00%
Blastoise       0.00%
Bulbasaur       0.00%
Butterfree      0.00%
Caterpie        0.00%
Chansey         0.00%
Charizard       0.00%
Charmander      100.00%
Charmeleon      0.00%
Clefable        0.00%
Clefairy        0.00%
Cloyster        0.00%
Cubone          0.00%
Dewgong         0.00%
Diglett         0.00%
Ditto           0.00%
Dodrio          0.00%
Doduo           0.00%
Dragonair       0.00%
Dragonite       0.00%
Dratini         0.00%
Drowzee         0.00%
Dugtrio         0.00%
Eevee           0.00%
Ekans           0.00%
Electabuzz      0.00%
Electrode       0.00%
Exeggcute       0.00%
Exeggutor       0.00%
Farfetchd       0.00%
Fearow          0.00%
Flareon         0.00%
Gastly          0.00%
Gengar          0.00%
Geodude         0.00%
Gloom           0.00%
Golbat          0.00%
Goldeen         0.00%
Golduck 